# Building Knowledge Graphs from Data

Knowledge graphs are built from **triples**: `(subject, predicate, object)`. The data source determines the extraction approach:

| Source Type | Examples | Approach |
|-------------|----------|----------|
| **Structured** | CSV, SQL, APIs | Direct conversion to triples |
| **Semi-structured** | JSON, XML, infoboxes | Schema mapping |
| **Unstructured** | Free text, documents | NLP/LLM extraction (see notebooks 03 & 04) |

This notebook covers building KGs from **structured** and **semi-structured** data.

In [ ]:
%pip install networkx matplotlib pandas -q

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

## 1. From Structured Data (CSV / Tables)

When data is already in a well-defined schema (database tables, CSVs), each row can be directly converted to triples:

```
Row: Alice | Engineer | AI Research | TechCorp | Bob
 →  (Alice, has_role, Engineer)
 →  (Alice, in_department, AI Research)
 →  (Alice, works_at, TechCorp)
 →  (Alice, reports_to, Bob)
```

In [ ]:
# Sample employee data
df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "Role": ["Engineer", "Director", "VP", "Analyst", "Researcher"],
    "Department": ["AI Research", "AI Research", "Engineering", "Data Science", "AI Research"],
    "Company": ["TechCorp", "TechCorp", "TechCorp", "TechCorp", "TechCorp"],
    "Reports_To": ["Bob", "Carol", None, "Bob", "Alice"]
})
df

In [ ]:
def dataframe_to_triples(df):
    """Convert each row into knowledge graph triples."""
    triples = []
    for _, row in df.iterrows():
        name = row["Name"]
        triples.append((name, "has_role", row["Role"]))
        triples.append((name, "in_department", row["Department"]))
        triples.append((name, "works_at", row["Company"]))
        if pd.notna(row["Reports_To"]):
            triples.append((name, "reports_to", row["Reports_To"]))
    return triples

triples = dataframe_to_triples(df)
print(f"Extracted {len(triples)} triples:\n")
for s, p, o in triples:
    print(f"  ({s}) —{p}→ ({o})")

In [ ]:
# Build and visualize the knowledge graph
G_emp = nx.DiGraph()
for s, p, o in triples:
    G_emp.add_edge(s, o, relation=p)

# Categorize nodes for coloring
people = set(df["Name"])
roles = set(df["Role"])
depts = set(df["Department"])
companies = set(df["Company"])

color_map = []
for node in G_emp.nodes():
    if node in people:
        color_map.append("#FF9999")
    elif node in roles:
        color_map.append("#99CCFF")
    elif node in depts:
        color_map.append("#99FF99")
    else:
        color_map.append("#FFCC99")

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G_emp, seed=42, k=1.8)
nx.draw(G_emp, pos, with_labels=True, node_color=color_map,
        node_size=2500, font_size=9, font_weight="bold",
        arrows=True, arrowsize=15, edge_color="gray")

edge_labels = nx.get_edge_attributes(G_emp, "relation")
nx.draw_networkx_edge_labels(G_emp, pos, edge_labels=edge_labels, font_size=7)

legend_handles = [
    mpatches.Patch(color="#FF9999", label="People"),
    mpatches.Patch(color="#99CCFF", label="Roles"),
    mpatches.Patch(color="#99FF99", label="Departments"),
    mpatches.Patch(color="#FFCC99", label="Company"),
]
plt.legend(handles=legend_handles, loc="upper left", fontsize=10)
plt.title("Knowledge Graph from Employee Table", fontsize=14)
plt.tight_layout()
plt.show()

## 2. From Semi-Structured Data (JSON)

Semi-structured data (JSON, XML, API responses) requires **schema mapping** — deciding which fields become entities, which become relationships, and how to normalize names.

In [ ]:
# Sample research paper data (as from an API or JSON file)
papers = [
    {
        "title": "Attention Is All You Need",
        "authors": ["Vaswani", "Shazeer", "Parmar"],
        "year": 2017,
        "institution": "Google Brain",
        "topics": ["transformers", "NLP"]
    },
    {
        "title": "BERT",
        "authors": ["Devlin", "Chang"],
        "year": 2018,
        "institution": "Google AI",
        "topics": ["NLP", "pre-training"]
    },
    {
        "title": "GPT-3",
        "authors": ["Brown", "Mann"],
        "year": 2020,
        "institution": "OpenAI",
        "topics": ["language models", "few-shot"]
    },
    {
        "title": "Graph Attention Networks",
        "authors": ["Velickovic", "Cucurull"],
        "year": 2018,
        "institution": "DeepMind",
        "topics": ["graph neural networks", "transformers"]
    }
]

In [ ]:
def json_to_triples(papers):
    """Extract triples from paper JSON records."""
    triples = []
    for paper in papers:
        title = paper["title"]
        for author in paper["authors"]:
            triples.append((title, "written_by", author))
            triples.append((author, "affiliated_with", paper["institution"]))
        triples.append((title, "published_by", paper["institution"]))
        for topic in paper["topics"]:
            triples.append((title, "covers_topic", topic))
    return triples

paper_triples = json_to_triples(papers)
print(f"Extracted {len(paper_triples)} triples:\n")
for s, p, o in paper_triples[:10]:
    print(f"  ({s}) —{p}→ ({o})")
print(f"  ... and {len(paper_triples) - 10} more")

In [ ]:
# Build and visualize the paper knowledge graph
G_papers = nx.DiGraph()
for s, p, o in paper_triples:
    G_papers.add_edge(s, o, relation=p)

# Categorize nodes
paper_titles = {p["title"] for p in papers}
authors = {a for p in papers for a in p["authors"]}
institutions = {p["institution"] for p in papers}
topics = {t for p in papers for t in p["topics"]}

color_map = []
for node in G_papers.nodes():
    if node in paper_titles:
        color_map.append("#FF9999")
    elif node in authors:
        color_map.append("#99CCFF")
    elif node in institutions:
        color_map.append("#99FF99")
    else:
        color_map.append("#FFCC99")

plt.figure(figsize=(15, 10))
pos = nx.spring_layout(G_papers, seed=42, k=1.5)
nx.draw(G_papers, pos, with_labels=True, node_color=color_map,
        node_size=2500, font_size=8, font_weight="bold",
        arrows=True, arrowsize=12, edge_color="gray")

edge_labels = nx.get_edge_attributes(G_papers, "relation")
nx.draw_networkx_edge_labels(G_papers, pos, edge_labels=edge_labels, font_size=7)

legend_handles = [
    mpatches.Patch(color="#FF9999", label="Papers"),
    mpatches.Patch(color="#99CCFF", label="Authors"),
    mpatches.Patch(color="#99FF99", label="Institutions"),
    mpatches.Patch(color="#FFCC99", label="Topics"),
]
plt.legend(handles=legend_handles, loc="upper left", fontsize=10)
plt.title("Knowledge Graph from Research Papers (JSON)", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Querying the Knowledge Graph

Once built, we can traverse the graph to answer questions — the same way an agent would query a KG for retrieval-augmented reasoning.

In [ ]:
# Question: "Which authors are connected through shared topics?"
# Strategy: find authors who wrote papers covering the same topic

from collections import defaultdict

topic_to_authors = defaultdict(set)
for paper in papers:
    for topic in paper["topics"]:
        for author in paper["authors"]:
            topic_to_authors[topic].add(author)

print("Authors connected through shared topics:\n")
for topic, author_set in topic_to_authors.items():
    if len(author_set) > 1:
        print(f"  Topic '{topic}': {', '.join(sorted(author_set))}")

In [ ]:
# Question: "What topics does Google Brain research?"
# Traverse: institution ← published_by ← paper → covers_topic → topic

institution = "Google Brain"
predecessors = list(G_papers.predecessors(institution))  # papers pointing to this institution
gb_topics = set()
for paper_node in predecessors:
    if paper_node in paper_titles:
        for neighbor in G_papers.successors(paper_node):
            if neighbor in topics:
                gb_topics.add(neighbor)

print(f"Topics researched by {institution}: {', '.join(gb_topics)}")

In [ ]:
# Question: "Find the path between Vaswani and the topic 'NLP'"
try:
    path = nx.shortest_path(G_papers, "Vaswani", "NLP")
    print("Path from Vaswani to NLP:")
    for i in range(len(path) - 1):
        edge_data = G_papers.get_edge_data(path[i], path[i+1])
        if edge_data:
            print(f"  ({path[i]}) —{edge_data['relation']}→ ({path[i+1]})")
        else:
            # Check reverse direction
            edge_data = G_papers.get_edge_data(path[i+1], path[i])
            if edge_data:
                print(f"  ({path[i]}) ←{edge_data['relation']}— ({path[i+1]})")
except nx.NetworkXNoPath:
    print("No path found. Trying undirected version...")
    G_undir = G_papers.to_undirected()
    path = nx.shortest_path(G_undir, "Vaswani", "NLP")
    print(f"  Path (undirected): {' → '.join(path)}")

## 4. Building a KG for an Agentic AI System

In an agentic system, the KG tracks **agents and their capabilities**, **tasks and their requirements**, **documents**, and **tools**. The graph enables the system to match agents to tasks by comparing capabilities against requirements.

In [ ]:
# Define the agentic system components
agent_data = [
    {"id": "SummaryAgent", "type": "LLMAgent", "capabilities": ["summarization", "writing"]},
    {"id": "ResearchAgent", "type": "RetrievalAgent", "capabilities": ["web_search", "document_retrieval"]},
    {"id": "CodeAgent", "type": "LLMAgent", "capabilities": ["code_generation", "debugging"]},
    {"id": "DataAgent", "type": "AnalyticsAgent", "capabilities": ["data_analysis", "visualization"]},
]

task_data = [
    {"id": "Task:SummarizePaper", "requires": ["summarization", "document_retrieval"]},
    {"id": "Task:BuildDashboard", "requires": ["data_analysis", "visualization", "code_generation"]},
    {"id": "Task:LitReview", "requires": ["web_search", "summarization"]},
]

# Build the KG
G_agentic = nx.DiGraph()

all_agents, all_tasks, all_capabilities = [], [], set()

for agent in agent_data:
    all_agents.append(agent["id"])
    G_agentic.add_edge(agent["id"], agent["type"], relation="is_type")
    for cap in agent["capabilities"]:
        G_agentic.add_edge(agent["id"], cap, relation="CAPABLE_OF")
        all_capabilities.add(cap)

for task in task_data:
    all_tasks.append(task["id"])
    for req in task["requires"]:
        G_agentic.add_edge(task["id"], req, relation="REQUIRES")
        all_capabilities.add(req)

# Color by type
agent_types = {a["type"] for a in agent_data}
color_map = []
for node in G_agentic.nodes():
    if node in all_agents:
        color_map.append("#FF9999")
    elif node in all_tasks:
        color_map.append("#99CCFF")
    elif node in all_capabilities:
        color_map.append("#FFCC99")
    else:
        color_map.append("#D9D9D9")

plt.figure(figsize=(15, 10))
pos = nx.spring_layout(G_agentic, seed=42, k=1.8)
nx.draw(G_agentic, pos, with_labels=True, node_color=color_map,
        node_size=2500, font_size=8, font_weight="bold",
        arrows=True, arrowsize=12, edge_color="gray")

edge_labels = nx.get_edge_attributes(G_agentic, "relation")
nx.draw_networkx_edge_labels(G_agentic, pos, edge_labels=edge_labels, font_size=7)

legend_handles = [
    mpatches.Patch(color="#FF9999", label="Agents"),
    mpatches.Patch(color="#99CCFF", label="Tasks"),
    mpatches.Patch(color="#FFCC99", label="Capabilities"),
    mpatches.Patch(color="#D9D9D9", label="Agent Types"),
]
plt.legend(handles=legend_handles, loc="upper left", fontsize=10)
plt.title("Agentic AI System Knowledge Graph", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Agent matching: "Find agents that can handle each task"
# An agent can handle a task if it has capabilities that overlap with requirements

agent_caps = {a["id"]: set(a["capabilities"]) for a in agent_data}
task_reqs = {t["id"]: set(t["requires"]) for t in task_data}

print("=== Task-Agent Matching ===")
for task_id, reqs in task_reqs.items():
    print(f"\n{task_id} (requires: {', '.join(reqs)})")
    for agent_id, caps in agent_caps.items():
        overlap = caps & reqs
        coverage = len(overlap) / len(reqs)
        if overlap:
            print(f"  {agent_id}: covers {coverage:.0%} — matches: {', '.join(overlap)}")

## Key Takeaways

- **Structured data** (CSV, tables) → direct triple conversion by mapping columns to predicates
- **Semi-structured data** (JSON, XML) → requires schema mapping to align fields with ontology terms
- KGs enable **structured querying** — finding paths, common neighbors, and answering multi-hop questions
- In agentic systems, KGs enable **capability-based task routing** by matching agent capabilities to task requirements
- Graph structure reveals **relationships not obvious in tabular data** (e.g., which authors are connected through shared topics)